In [1]:
import matplotlib
matplotlib.use("Agg")
import numpy as np
import matplotlib.pyplot as plt
import h5py

import importlib
import os
import sys
import re
import glob

In [2]:
path_openket = "//home//ultrxvioletx//GitHub//openket"
if path_openket not in sys.path:
    sys.path.append(path_openket)
# elimina el caché de func.py para que no lea los datos anteriores
if 'func' in sys.modules:
    del sys.modules['func']
if os.path.exists('func.py'):
    os.remove('func.py')

from openket.core.diracobject import Ket, Bra, Operator, AnnihilationOperator, CreationOperator
from openket.core.evolution import build_ode, sym2num, init_state
from openket.core.metrics import dag, comm, ptrace, trace, normalize, sub_qexpr, op2dict, qmatrix

import style
importlib.reload(style)
from style import set_style, colores, format_ax
set_style()

2026-03-22 01:10:16,894 - openket - INFO - openket v0.1.0 initialized successfully.


In [3]:
def convergencia(tiempo, fotones, tol=1e-3, porcentaje=0.1):
    npoints = int(len(tiempo) * porcentaje)
    if npoints < 2:
        return False, np.nan # no hay suficientes puntos
    fotones = fotones[-npoints:]
    tiempo = tiempo[-npoints:]
    df = np.gradient(fotones, tiempo)
    df_mean = np.mean(np.abs(df))
    return df_mean < tol, df_mean
def procesafile(path):
    global detunings, Nss, non_converged, attrs
    attrs = {}
    detunings = []
    Nss = []
    
    non_converged = []
    files = glob.glob(os.path.join(path,'*.h5'))
    for i,file in enumerate(files):
        with h5py.File(file, 'r') as f:
            dataset = os.path.basename(file).replace('.h5', '')
            if i == 0:
                attrs = dict(f[dataset].attrs)

            detuning = f[dataset].attrs[f'detuning']
            tt = f[dataset].attrs['t']
            tiempo = np.linspace(float(tt[0]), float(tt[1]), int(tt[2]))
            
            if dataset not in f:
                print(f"  no se encontró el dataset '{dataset}' en el archivo. Saltando.")
                continue

            detunings.append(detuning)
            rho = f[dataset][:]
            N_expect = rho[:, 0]

            # promediamos el último 25% de los puntos
            npoints = int(rho.shape[0] * 0.25)
            Nss.append(np.mean(N_expect[-npoints:]))
            
            is_converged, derivative = convergencia(tiempo, N_expect)
            if not is_converged:
                non_converged.append([detuning, np.mean(N_expect[-npoints:])])
    
    # ordena por detuning para mejorar la gráfica
    detunings = np.array(detunings)
    sorti = np.argsort(detunings)
    
    detunings = detunings[sorti]
    Nss = np.array(Nss)[sorti]

In [6]:
path = os.path.join("detunings","cavidad")
procesafile(path)

cut=10
plt.plot(detunings[cut:-cut], Nss[cut:-cut], 's-', markersize=5, linewidth=1, color=colores["fotones_ss"])
if non_converged:
    dtn = [ele[0] for ele in non_converged]
    nmean = [ele[1] for ele in non_converged]
    #plt.scatter(dtn, nmean, color="red")
plt.axvline(0, color="gray", linestyle="--", linewidth=1)
plt.xlabel(r"$\Delta / \kappa$")
plt.ylabel(r"Número medio de fotones $\langle N \rangle_{ss}$")
ax = plt.gca()
xval=3.5
format_ax(ax, xstep=1, ystep=2, ymax=max(Nss)+1, xlim=(-xval,xval))
plt.savefig(f"../figuras/fig:deltas_fotones.png")
plt.close()

for file in os.listdir("."):
    if file.endswith(".py") and file != "style.py":
        os.remove(file)

In [5]:
attrs

{'detuning': np.float64(3.6),
 'id': np.int32(77),
 'kappa': np.float64(1.0),
 'n': np.int32(10),
 'rabi_p': np.float64(1.0),
 't': array(['0', '10.0', '1000'], dtype=object)}